# 00 — Data & Split (proposal 3.1, 3.3.1)

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

## Load + preprocess
Apply the filtering pipeline in proposal order. Assert class counts before freezing.

In [3]:
from src import data

clean = data.load_or_build_clean()          # builds clean.parquet once
print(clean.shape)
clean[config.LABEL_COL].value_counts()

(159288, 12)


model
gpt4            13363
chatgpt         13331
cohere          13312
cohere-chat     13305
gpt3            13275
human           13269
llama-chat      13249
mpt-chat        13249
mistral-chat    13247
mpt             13246
mistral         13238
gpt2            13204
Name: count, dtype: int64

## Grouped stratified split
Grouped by `source_id`, 70/15/15. Freeze the indices — never rebuild.

In [4]:
splits = data.load_or_build_splits(clean)     # writes artifacts/split_indices.json
{k: len(v) for k, v in splits.items()}

{'train': 113772, 'val': 22760, 'test': 22756}

## Freeze check
No `source_id` may cross splits; class balance roughly preserved.

In [5]:
# assertions run inside load_or_build_splits; eyeball balance here
for k, idx in splits.items():
    print(k, clean.iloc[idx][config.LABEL_COL].value_counts(normalize=True).round(3).to_dict())

train {'gpt4': 0.084, 'chatgpt': 0.084, 'cohere': 0.084, 'cohere-chat': 0.084, 'gpt3': 0.083, 'human': 0.083, 'llama-chat': 0.083, 'mpt': 0.083, 'mpt-chat': 0.083, 'mistral-chat': 0.083, 'mistral': 0.083, 'gpt2': 0.083}
val {'gpt4': 0.084, 'chatgpt': 0.084, 'cohere-chat': 0.084, 'cohere': 0.084, 'human': 0.083, 'gpt3': 0.083, 'llama-chat': 0.083, 'mpt-chat': 0.083, 'mistral': 0.083, 'mistral-chat': 0.083, 'mpt': 0.083, 'gpt2': 0.083}
test {'gpt4': 0.084, 'chatgpt': 0.084, 'cohere': 0.084, 'cohere-chat': 0.084, 'gpt3': 0.083, 'human': 0.083, 'llama-chat': 0.083, 'mpt-chat': 0.083, 'mistral-chat': 0.083, 'mpt': 0.083, 'mistral': 0.083, 'gpt2': 0.083}
